# 04 · Router de consultas y datos estructurados (SQL)

**Idea:** en producción casi nunca eliges *una* familia de RAG: **clasificas la pregunta y la envías al método correcto**.
- «¿Cuántas unidades de electrónica se vendieron en Cusco en septiembre?» → **SQL** (un cálculo, no una búsqueda).
- «ERR-5021» → **palabra clave**.
- «¿Quién aprueba los viajes de la jefa de Pagos?» → **grafo**.
- «¿Puedo trabajar desde mi casa?» → **vectorial/híbrido**.

En este notebook:
1. Compruebas por qué el RAG de texto **no sirve** para agregaciones.
2. Entrenas un **router** pequeño y lo mides con preguntas que no vio.
3. Conectas el router con un **SQL de juguete** (con una lista blanca de columnas, como haría un text-to-SQL seguro).

## Glosario · router y SQL

Lo que necesitas para leer este notebook (puedes volver aquí cuando una palabra no te suene):

- **Embedding:** lista de números que representa el *significado* de un texto; textos con sentido parecido quedan «cerca», aunque no compartan palabras.
- **Similitud coseno:** qué tan cerca están dos embeddings (cerca de 1 = parecidos; cerca de 0 = nada que ver).
- **BM25:** puntaje por *palabras exactas*: cuenta qué documentos contienen las palabras de la pregunta y valora más las raras (`ERR-5012`) que las comunes. No entiende sinónimos; si no comparte ninguna palabra, no devuelve nada. Su puntaje no tiene tope y no es comparable con el coseno.
- **Top-k:** los *k* primeros resultados del ranking (top-3 = los tres mejores).
- **Híbrido calibrado:** mezcla `α · coseno + (1 − α) · BM25` (ambos llevados a 0–1), con α elegido en el set de ajuste. Se explica en el notebook 01; aquí se reutiliza.
- **Router:** clasificador que decide a qué método (SQL, grafo, código, texto) manda cada pregunta.
- **TF-IDF de caracteres:** cuenta trozos de letras de la pregunta y pesa más los poco comunes.
- **Regresión logística:** clasificador sencillo que estima la probabilidad de cada ruta a partir de esos conteos.
- **Text-to-SQL:** traducir la pregunta a una consulta SQL.
- **Lista blanca y SQL parametrizado:** la consulta se arma solo con columnas y valores permitidos, y los valores van como parámetros, nunca pegando el texto del usuario (evita la *inyección de SQL*).
- **Usuario de solo lectura:** una cuenta de base de datos que puede consultar pero no modificar nada.
- **Exactitud:** fracción de preguntas que el router manda a la ruta correcta.

## Los datos de este notebook (qué es cada campo)

| Conjunto | Qué es | Campos de cada elemento |
|---|---|---|
| tabla de ventas (`build_sales()`) | 27 filas sintéticas | `tienda`, `categoria`, `mes`, `unidades` (vendidas) y `monto` (soles = unidades × precio) |
| `ENTRENAMIENTO` | 28 frases con la ruta correcta, para entrenar el router | `(pregunta, ruta)`: ruta ∈ sql · grafo · codigo · texto |
| `DOCS`, `QUERIES`, `TRIPLES` | los mismos datos de los notebooks 01 y 03 (para las rutas texto y grafo) | ver el notebook 01 y el 03 |

In [1]:
import importlib.util, subprocess, sys

for paquete, modulo in [("rank-bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers"), ("scikit-learn", "sklearn")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", paquete])
print("Entorno listo")

Entorno listo


In [2]:
#@title Datos sintéticos de «Andina Retail» (no necesitas leer esta celda) { display-mode: "form" }
"""Corpus sintético de «Andina Retail» para los notebooks del taller.

Se inserta tal cual en cada notebook (celda oculta con #@title), así cada uno corre solo en Colab.
Todo es inventado: las métricas que salgan ilustran un criterio, no miden producción.
"""

# ---------------------------------------------------------------- documentos
# ESQUEMA · DOCS (el corpus: 42 «documentos» cortos, cada uno es una tupla de 4 campos)
#   (id, título, texto, tipo)
#   id     código único del documento. La letra dice de qué familia es:
#            P = política interna · T = técnico (código exacto) · D = «distractor» (código casi idéntico a uno T)
#            E = entidades y relaciones (personas, equipos, sistemas; sirven para el notebook de grafo)
#   título nombre corto del documento (se indexa junto con el texto).
#   texto  contenido que se busca y que se le entregaría al modelo (1–3 frases).
#   tipo   categoría temática: politica · tecnico · producto · contrato · organizacion · incidente.
#          No se usa para buscar; solo describe el documento.
# Ejemplo: ("P01", "Vacaciones", "Los colaboradores acumulan 30 días…", "politica")
DOCS = [
    # Políticas (preguntas semánticas, con paráfrasis)
    ("P01", "Vacaciones", "Los colaboradores acumulan 30 días calendario de descanso remunerado por año de servicio. El periodo puede fraccionarse en bloques de mínimo 7 días con aprobación del jefe directo.", "politica"),
    ("P02", "Teletrabajo", "Se permite el trabajo remoto hasta tres días por semana. El día presencial obligatorio es el martes. Se requiere conexión VPN corporativa y equipo entregado por la empresa.", "politica"),
    ("P03", "Devoluciones", "El cliente puede devolver un producto dentro de 30 días desde la compra, con boleta y empaque sellado. Los productos de electrónica tienen un plazo de 15 días. El reembolso se realiza al mismo medio de pago.", "politica"),
    ("P04", "Viáticos", "Los gastos de viaje se reembolsan con comprobante dentro de 10 días hábiles. El tope diario de alimentación es 120 soles. Los viajes internacionales requieren aprobación de la gerencia del área.", "politica"),
    ("P05", "Licencias", "La licencia por paternidad es de 10 días calendario y la de maternidad de 98 días. Se solicita a Recursos Humanos con el certificado médico correspondiente.", "politica"),
    ("P06", "Seguridad de la información", "Las claves de acceso deben tener 14 caracteres y cambiarse cada 90 días. Todo acceso a sistemas internos exige verificación en dos pasos (MFA). Está prohibido compartir credenciales.", "politica"),
    ("P07", "Código de conducta: obsequios", "No se aceptan obsequios, invitaciones ni atenciones de proveedores con valor mayor a 100 soles. Todo ofrecimiento debe reportarse al área de Cumplimiento.", "politica"),
    ("P08", "Proceso de compras", "Toda compra sobre 5,000 soles exige tres cotizaciones comparables y la aprobación del comité de compras. Las compras menores se tramitan con una sola cotización.", "politica"),
    ("P09", "Capacitación", "Cada colaborador dispone de un presupuesto anual de 2,000 soles para cursos, certificaciones y congresos, previa autorización de su jefe directo.", "politica"),
    ("P10", "Horario y tardanzas", "El horario de oficina es de 9:00 a 18:00. Las tardanzas acumuladas superiores a 60 minutos en el mes se descuentan de la remuneración y tres tardanzas generan una amonestación escrita.", "politica"),
    # Técnicos con códigos exactos (palabra clave)
    ("T01", "ERR-5021 pasarela de pagos", "ERR-5021: fallo de conexión con la pasarela de pagos PayHub. Reintentar tras 30 segundos; si persiste, escalar al equipo Pagos con el identificador de la transacción.", "tecnico"),
    ("T02", "ERR-7714 inventario", "ERR-7714: inventario desincronizado entre la tienda y el almacén central. Ejecutar la reconciliación nocturna con el job REC-NIGHT-02 y validar el stock al día siguiente.", "tecnico"),
    ("T03", "ERR-1003 sesión", "ERR-1003: el token de sesión de la aplicación móvil expiró. El usuario debe volver a iniciar sesión; no requiere escalamiento.", "tecnico"),
    ("T04", "SKU-48213 licuadora", "SKU-48213: Licuadora Oster ProBlend de 1.5 litros, línea hogar. Garantía de 24 meses con boleta de compra.", "producto"),
    ("T05", "SKU-90117 laptop", "SKU-90117: Laptop Lenovo IdeaPad 5 con 16 GB de RAM, línea electrónica. Garantía de 12 meses con boleta de compra.", "producto"),
    ("T06", "SKU-33502 refrigeradora", "SKU-33502: Refrigeradora Samsung de 380 litros con tecnología inverter, línea blanca. Garantía de 36 meses con boleta de compra.", "producto"),
    ("T07", "RB-204 certificados TLS", "RB-204: procedimiento para rotar los certificados TLS del balanceador. Se ejecuta en ventana de mantenimiento, con aprobación del responsable de Infraestructura y validación posterior de los endpoints.", "tecnico"),
    ("T08", "Contrato CT-2024-0087", "CT-2024-0087: contrato de transporte con el proveedor Logística Andes, vigente hasta diciembre de 2026, con penalidad del 2% por entregas tardías.", "contrato"),
    # Distractores: códigos casi idénticos, como en cualquier catálogo o log real
    ("D01", "ERR-5012 pasarela", "ERR-5012: tiempo de espera agotado al consultar el estado de un pago en la pasarela de pagos PayHub. Reintentar la consulta una vez y validar la transacción manualmente.", "tecnico"),
    ("D02", "ERR-5201 conciliación", "ERR-5201: diferencia en la conciliación bancaria diaria de la pasarela de pagos. Revisar el reporte de conciliación y escalar al equipo de Finanzas Digital.", "tecnico"),
    ("D03", "ERR-7741 stock", "ERR-7741: el stock reservado de un pedido supera el inventario disponible en tienda. Liberar la reserva y volver a calcular la disponibilidad.", "tecnico"),
    ("D04", "ERR-1030 sesión web", "ERR-1030: la sesión del portal web de clientes fue cerrada por inactividad. El usuario debe iniciar sesión nuevamente.", "tecnico"),
    ("D05", "SKU-48231 licuadora", "SKU-48231: Licuadora Oster ProBlend de 1.2 litros, línea hogar. Garantía de 12 meses con boleta de compra.", "producto"),
    ("D06", "SKU-48123 batidora", "SKU-48123: Batidora de pedestal Oster de 5 velocidades, línea hogar. Garantía de 18 meses con boleta de compra.", "producto"),
    ("D07", "SKU-90171 laptop", "SKU-90171: Laptop Lenovo IdeaPad 3 con 8 GB de RAM, línea electrónica. Garantía de 12 meses con boleta de compra.", "producto"),
    ("D08", "SKU-33205 refrigeradora", "SKU-33205: Refrigeradora Samsung de 300 litros, línea blanca. Garantía de 24 meses con boleta de compra.", "producto"),
    ("D09", "RB-240 certificados", "RB-240: procedimiento para renovar los certificados del correo corporativo. Se ejecuta fuera del horario de oficina y lo valida el equipo de Soporte.", "tecnico"),
    ("D10", "Contrato CT-2024-0078", "CT-2024-0078: contrato de limpieza con el proveedor Servi Limpio, vigente hasta junio de 2027, con penalidad del 1% por incumplimiento.", "contrato"),
    ("D11", "Contrato CT-2023-0087", "CT-2023-0087: contrato de vigilancia con el proveedor Seguridad Altiplano, vigente hasta marzo de 2026, renovación pendiente.", "contrato"),
    ("D12", "Job REC-NIGHT-01", "REC-NIGHT-01: job de reconciliación nocturna de ventas entre tiendas y contabilidad. Se ejecuta a las 02:00 y envía un reporte al área de Finanzas.", "tecnico"),
    # Entidades y relaciones (multi-hop)
    ("E01", "Equipo Pagos", "El equipo Pagos, liderado por Rosa Quispe, mantiene la pasarela de pagos PayHub.", "organizacion"),
    ("E02", "Equipo Finanzas Digital", "El sistema de facturación FacturaPlus es mantenido por el equipo Finanzas Digital, liderado por Marco Salas.", "organizacion"),
    ("E03", "Marco Salas", "Marco Salas reporta a la gerente de Tecnología, Lucía Paredes.", "organizacion"),
    ("E04", "Rosa Quispe", "Rosa Quispe reporta a la gerente de Tecnología, Lucía Paredes.", "organizacion"),
    ("E05", "Lucía Paredes", "Lucía Paredes, gerente de Tecnología, aprueba los viajes internacionales del área de Tecnología.", "organizacion"),
    ("E06", "PayHub y antifraude", "PayHub depende del servicio antifraude RiskGuard, operado por el proveedor externo SafeNet Perú.", "organizacion"),
    ("E07", "FacturaPlus y SUNAT", "FacturaPlus emite los comprobantes electrónicos a través del operador FactuRed, certificado ante SUNAT.", "organizacion"),
    ("E08", "Cobertura del contrato de transporte", "El contrato CT-2024-0087 con Logística Andes cubre el transporte hacia las tiendas de Arequipa y Cusco.", "organizacion"),
    ("E09", "Gestión de Logística Andes", "El proveedor Logística Andes es gestionado por el área de Compras, a cargo de Daniela Ríos.", "organizacion"),
    ("E10", "Daniela Ríos", "Daniela Ríos reporta al gerente de Operaciones, Jorge Medina.", "organizacion"),
    ("E11", "Tienda Arequipa", "La tienda de Arequipa es administrada por Carmen Valdivia, quien reporta a Jorge Medina.", "organizacion"),
    ("E12", "Incidente 3 de septiembre", "El 3 de septiembre una caída del servicio RiskGuard provocó una ráfaga de errores ERR-5021 en PayHub durante 40 minutos.", "incidente"),
]

# ------------------------------------------------- consultas con juez (doc-level)
# tipo: semantica (paráfrasis), exacta (código/ID), mixta (ambas), multihop (necesita varios docs)
# ESQUEMA · QUERIES (las 33 preguntas con respuesta conocida: el «golden set»; tuplas de 3 campos)
#   (pregunta, tipo, relevantes)
#   pregunta   texto tal como lo escribiría un usuario.
#   tipo       semantica (parafrasea; casi no comparte palabras con el documento) ·
#              exacta (un código o identificador) · mixta (lenguaje natural + código).
#   relevantes conjunto con los id de los documentos correctos. Es el «juez»: una pregunta acierta si alguno
#              de ellos aparece arriba en el ranking.
# Ejemplo: ("¿Puedo trabajar desde mi casa?", "semantica", {"P02"})  → el correcto es «Teletrabajo».
# Más abajo, dividir() parte estas 33 en 17 de ajuste (dev) y 16 de prueba (test).
QUERIES = [
    # semánticas: casi sin palabras en común con el documento
    ("¿Cuántos días libres pagados tengo cada año?", "semantica", {"P01"}),
    ("¿Puedo trabajar desde mi casa?", "semantica", {"P02"}),
    ("Compré una tele y no me gustó, ¿me devuelven la plata?", "semantica", {"P03"}),
    ("¿Me pagan lo que gasto cuando viajo por trabajo?", "semantica", {"P04"}),
    ("¿Cómo protejo mis accesos a los sistemas?", "semantica", {"P06"}),
    ("¿Puedo recibir un regalo de un proveedor?", "semantica", {"P07"}),
    ("¿Hay dinero para hacer cursos y certificaciones?", "semantica", {"P09"}),
    ("Llegué tarde varias veces, ¿qué consecuencias tiene?", "semantica", {"P10"}),
    ("¿Cuál es el plazo para cambiar un producto que no me sirvió?", "semantica", {"P03"}),
    ("¿Qué trámite hago para pedir descanso cuando nace mi hijo?", "semantica", {"P05"}),
    ("¿Cuántas ofertas de precio necesito antes de comprar algo caro?", "semantica", {"P08"}),
    ("¿A qué hora debo estar en la oficina?", "semantica", {"P10"}),
    # exactas: un código o identificador
    ("ERR-5021", "exacta", {"T01", "E12"}),
    ("SKU-48213", "exacta", {"T04"}),
    ("¿Qué significa ERR-7714?", "exacta", {"T02"}),
    ("CT-2024-0087", "exacta", {"T08", "E08"}),
    ("RB-204", "exacta", {"T07"}),
    ("REC-NIGHT-02", "exacta", {"T02"}),
    ("SKU-33502", "exacta", {"T06"}),
    ("ERR-1003", "exacta", {"T03"}),
    ("SKU-90117", "exacta", {"T05"}),
    ("ERR-5012", "exacta", {"D01"}),
    ("CT-2023-0087", "exacta", {"D11"}),
    ("SKU-48231", "exacta", {"D05"}),
    # mixtas: lenguaje natural + código
    ("¿Cuántos meses de garantía tiene la laptop SKU-90117?", "mixta", {"T05"}),
    ("Mi app del celular muestra ERR-1003, ¿qué hago?", "mixta", {"T03"}),
    ("Falla la pasarela de pagos con ERR-5021, ¿cómo lo resuelvo?", "mixta", {"T01"}),
    ("¿Hasta cuándo está vigente el contrato CT-2024-0087 de transporte?", "mixta", {"T08"}),
    ("Se desincronizó el inventario con ERR-7714, ¿qué proceso corre de noche?", "mixta", {"T02"}),
    ("¿Qué garantía tiene la batidora SKU-48123?", "mixta", {"D06"}),
    ("El portal web me cerró la sesión con ERR-1030, ¿es normal?", "mixta", {"D04"}),
    ("¿De qué trata el contrato de limpieza CT-2024-0078?", "mixta", {"D10"}),
    ("Hay diferencia en la conciliación bancaria con ERR-5201, ¿a quién escalo?", "mixta", {"D02"}),
]

# Preguntas multi-hop: docs que hay que juntar para poder responder + respuesta esperada
# ESQUEMA · MULTIHOP (6 preguntas que necesitan juntar varios documentos; tuplas de 3 campos)
#   (pregunta, documentos_necesarios, respuesta_esperada)
#   documentos_necesarios  id de TODOS los documentos E## que hay que unir para poder responder.
#   respuesta_esperada     dato final que sale de encadenar esos documentos.
# Aquí la pregunta NOMBRA las entidades que hay que encadenar (el texto suele bastar).
MULTIHOP = [
    ("¿Quién aprueba los viajes internacionales de la jefa del equipo que mantiene la pasarela de pagos?",
     {"E01", "E04", "E05"}, "Lucía Paredes"),
    ("¿Qué proveedor externo opera el servicio antifraude del que depende el sistema que mantiene el equipo de Rosa Quispe?",
     {"E01", "E06"}, "SafeNet Perú"),
    ("¿A quién reporta la persona que gestiona el proveedor que transporta mercadería a Arequipa?",
     {"E08", "E09", "E10"}, "Jorge Medina"),
    ("¿Quién es la jefa de la persona que lidera el equipo que mantiene el sistema de facturación?",
     {"E02", "E03"}, "Lucía Paredes"),
    ("¿Qué servicio antifraude provocó los errores ERR-5021 del 3 de septiembre?",
     {"E12", "E06"}, "RiskGuard"),
    ("¿Quién administra la tienda cuyo gerente de Operaciones también supervisa a quien gestiona Logística Andes?",
     {"E09", "E10", "E11"}, "Carmen Valdivia"),
]

# Puente oculto: el documento intermedio casi no comparte palabras con la pregunta
# ESQUEMA · MULTIHOP_PUENTE (3 preguntas; mismo formato que MULTIHOP)
# Diferencia: el documento intermedio («puente») casi no comparte palabras con la pregunta; por eso el texto
# se lo pierde y el grafo no.
MULTIHOP_PUENTE = [
    ("¿Quién tiene que autorizar el viaje internacional del líder del equipo de facturación?",
     {"E02", "E03", "E05"}, "Lucía Paredes"),
    ("¿Quién lidera el equipo que mantiene el sistema afectado el 3 de septiembre?",
     {"E12", "E01"}, "Rosa Quispe"),
    ("¿Quién es el superior de la persona que negocia con el proveedor que transporta a Cusco?",
     {"E08", "E09", "E10"}, "Jorge Medina"),
]

# Tripletas (sujeto, relación, objeto, doc_fuente). En un caso real las extrae un LLM de los documentos.
# ESQUEMA · TRIPLES (el grafo escrito a mano: una arista por tupla de 4 campos)
#   (sujeto, relación, objeto, doc_fuente)
#   sujeto y objeto  entidades (nodos del grafo): personas, equipos, sistemas, proveedores, lugares.
#   relación         verbo que las une (lidera, reporta_a, depende_de…).
#   doc_fuente       id del documento E## de donde sale esa relación (para citar cada salto).
# Ejemplo: ("Rosa Quispe", "reporta_a", "Lucía Paredes", "E04"). En un caso real las extrae un LLM.
TRIPLES = [
    ("Equipo Pagos", "lidera", "Rosa Quispe", "E01"),
    ("Equipo Pagos", "mantiene", "PayHub", "E01"),
    ("Equipo Finanzas Digital", "lidera", "Marco Salas", "E02"),
    ("Equipo Finanzas Digital", "mantiene", "FacturaPlus", "E02"),
    ("Marco Salas", "reporta_a", "Lucía Paredes", "E03"),
    ("Rosa Quispe", "reporta_a", "Lucía Paredes", "E04"),
    ("Lucía Paredes", "aprueba_viajes_de", "área de Tecnología", "E05"),
    ("PayHub", "depende_de", "RiskGuard", "E06"),
    ("RiskGuard", "operado_por", "SafeNet Perú", "E06"),
    ("FacturaPlus", "emite_vía", "FactuRed", "E07"),
    ("CT-2024-0087", "con_proveedor", "Logística Andes", "E08"),
    ("Logística Andes", "transporta_a", "Arequipa", "E08"),
    ("Logística Andes", "gestionado_por", "Daniela Ríos", "E09"),
    ("Daniela Ríos", "reporta_a", "Jorge Medina", "E10"),
    ("Carmen Valdivia", "administra", "tienda de Arequipa", "E11"),
    ("Carmen Valdivia", "reporta_a", "Jorge Medina", "E11"),
    ("RiskGuard", "causó_caída", "ERR-5021", "E12"),
    ("ERR-5021", "ocurrió_en", "PayHub", "E12"),
    ("Logística Andes", "transporta_a", "Cusco", "E08"),
]


# ESQUEMA · tabla de ventas (la crea build_sales(); una fila por tienda × categoría × mes = 27 filas)
#   tienda (Lima · Arequipa · Cusco) · categoria (hogar · electrónica · línea blanca) · mes (julio · agosto · septiembre)
#   unidades  cantidad vendida (número aleatorio fijo, semilla 7)
#   monto     ingresos en soles = unidades × precio de la categoría
def build_sales():
    """Tabla de ventas sintética y determinista, para el caso de SQL."""
    import numpy as np
    import pandas as pd
    rng = np.random.default_rng(7)
    filas = []
    for tienda in ["Lima", "Arequipa", "Cusco"]:
        for categoria in ["hogar", "electrónica", "línea blanca"]:
            for mes in ["julio", "agosto", "septiembre"]:
                unidades = int(rng.integers(40, 400))
                precio = {"hogar": 180, "electrónica": 2400, "línea blanca": 1900}[categoria]
                filas.append((tienda, categoria, mes, unidades, unidades * precio))
    return pd.DataFrame(filas, columns=["tienda", "categoria", "mes", "unidades", "monto"])

In [3]:
#@title Herramientas de búsqueda y evaluación (puedes dejarla cerrada) { display-mode: "form" }
import re, time, warnings
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

# «Palabras vacías»: palabras tan comunes que BM25 las ignora (no ayudan a distinguir un documento de otro).
STOP = set("""a al algo ante con como cada de del el ella en es esta este la las lo los me mi mis o para pero por
que qué se si su sus tengo tiene un una y ya cuántos cuántas cuándo cuál hay puedo hacer""".split())


def tok(texto):
    """Tokeniza en minúsculas, conserva códigos como ERR-5021 y quita palabras vacías."""
    # La expresión regular toma «palabras» (letras, números, tildes, ñ) y, si van unidas por guion, las deja como UNA sola
    # ficha: así «ERR-5021» no se parte en «err» y «5021». Es lo que permite que BM25 encuentre un código exacto.
    return [w for w in re.findall(r"[\wáéíóúñ]+(?:-[\w]+)*", texto.lower()) if w not in STOP]


# --- Los dos índices (se construyen una sola vez) ---
IDS = [d[0] for d in DOCS]                       # id de cada documento, en el mismo orden que los índices
TEXTOS = [f"{titulo}. {texto}" for _, titulo, texto, _ in DOCS]
# Índice por PALABRAS: BM25 guarda, para cada documento, sus fichas y cuán raras son en todo el corpus.
bm25 = BM25Okapi([tok(t) for t in TEXTOS])
# Índice por SIGNIFICADO: el modelo convierte cada documento en un vector (embedding). normalize_embeddings=True los deja
# con longitud 1; así luego el producto punto es directamente la similitud coseno.
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
EMB = modelo.encode(TEXTOS, normalize_embeddings=True)


def s_denso(q):
    """Similitud coseno entre la pregunta y cada documento."""
    return EMB @ modelo.encode([q], normalize_embeddings=True)[0]


def s_bm25(q):
    """Puntaje léxico BM25 de la pregunta contra cada documento."""
    return bm25.get_scores(tok(q))


def ranking(scores):
    """Ordena los ids de mayor a menor puntaje. Es la «lista ordenada» que cada método entrega."""
    return [IDS[i] for i in np.argsort(-scores, kind="stable")]  # desempate estable: mismo resultado en cualquier entorno


def r_denso(q):
    """Ranking vectorial: ordena TODOS los documentos por similitud coseno (siempre hay un orden, aunque sea flojo)."""
    return ranking(s_denso(q))


def r_bm25(q):
    """Solo los documentos que comparten algún término con la pregunta, como hace un buscador real.
    (Los de puntaje 0 no son «resultados»: ordenarlos solo agrega ruido arbitrario.)"""
    s = s_bm25(q)
    return [IDS[i] for i in np.argsort(-s, kind="stable") if s[i] > 0]


def minmax(x):
    """Lleva los puntajes a 0–1: el menor pasa a 0 y el mayor a 1 (por pregunta). Así coseno y BM25 se pueden mezclar.
    El 1e-9 evita dividir entre cero si todos los puntajes son iguales."""
    x = np.asarray(x, float)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


def convexa(q, alpha):
    """Híbrido por combinación de puntajes normalizados. alpha=1 es solo denso; alpha=0 es solo BM25."""
    # puntaje final = α · coseno(0–1) + (1 − α) · BM25(0–1). Es una mezcla de PUNTAJES (a diferencia de rrf, que mezcla posiciones).
    return ranking(alpha * minmax(s_denso(q)) + (1 - alpha) * minmax(s_bm25(q)))


def rrf(rankings, pesos=None, k=60):
    """Reciprocal Rank Fusion: fusiona listas ordenadas sin necesidad de normalizar puntajes."""
    pesos = pesos or [1] * len(rankings)          # «1:1»: todas las listas pesan igual
    acum = {}                                      # doc → puntaje acumulado
    for lista, w in zip(rankings, pesos):
        for pos, doc in enumerate(lista):          # pos empieza en 0 → el puesto real es pos + 1
            acum[doc] = acum.get(doc, 0) + w / (k + pos + 1)   # cada lista aporta 1 ÷ (60 + puesto); el puntaje original se ignora
    return sorted(acum, key=lambda d: -acum[d])   # mayor suma primero; un doc que solo está en una lista suma una sola vez


def evaluar(fn, consultas):
    """hit@3 (¿aparece un relevante entre los 3 primeros?) y MRR (1/posición del primer relevante), por tipo."""
    filas = []
    for q, tipo, relevantes in consultas:
        r = fn(q)                                   # ranking que devuelve el método para esta pregunta
        # puesto (desde 1) del PRIMER documento relevante; None si no aparece en la lista
        pos = next((i + 1 for i, d in enumerate(r) if d in relevantes), None)   # None = no apareció
        # por pregunta: hit@3 = 1 si pos ≤ 3, si no 0 · MRR = 1 ÷ pos (0 si no apareció)
        filas.append((tipo, float(pos is not None and pos <= 3), 1 / pos if pos else 0.0))
    df = pd.DataFrame(filas, columns=["tipo", "hit@3", "MRR"])
    out = df.groupby("tipo").mean()                 # promedio de las preguntas de cada tipo (semántica, exacta, mixta)
    out.loc["PROMEDIO TIPOS"] = out.mean()          # promedio de los TRES tipos (no de las preguntas sueltas)
    return out.round(2)


def dividir(consultas):
    """Separa cada tipo de consulta alternando: la mitad para ajustar (dev) y la otra para probar (test)."""
    # Dentro de cada tipo se va alternando: 1.ª → ajuste, 2.ª → prueba, 3.ª → ajuste… (así ambos conjuntos tienen de todo).
    cont, dev, test = {}, [], []
    for c in consultas:
        i = cont.get(c[1], 0)
        cont[c[1]] = i + 1
        (dev if i % 2 == 0 else test).append(c)
    return dev, test


print(f"{len(DOCS)} documentos · {len(QUERIES)} consultas con juez · modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2")

42 documentos · 33 consultas con juez · modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2


## 1. El problema: los embeddings no suman

Convertimos cada fila de ventas en una «frase» y la indexamos como si fuera un documento.

In [4]:
import sqlite3

ventas = build_sales()
display(ventas.head(4))

# El «RAG de texto» sobre una tabla: cada fila se convierte en una frase y se indexa con embeddings.
frases = [f"En {r.mes}, la tienda de {r.tienda} vendió {r.unidades} unidades de {r.categoria} por {r.monto} soles."
          for r in ventas.itertuples()]
emb_filas = modelo.encode(frases, normalize_embeddings=True)

pregunta = "¿Cuántas unidades de electrónica se vendieron en septiembre en total?"
# Se recuperan las 3 filas más parecidas a la pregunta (top-3), como hace cualquier RAG.
sim = emb_filas @ modelo.encode([pregunta], normalize_embeddings=True)[0]
top = np.argsort(-sim)[:3]
print("Filas que trae el RAG de texto (top-3):")
for i in top:
    print("  ·", frases[i])

# rag = suma de SOLO las 3 filas que trajo; real = suma de TODAS las filas que cumplen la condición (lo que haría SQL).
real = int(ventas.query("categoria == 'electrónica' and mes == 'septiembre'")["unidades"].sum())
rag = int(ventas.iloc[top]["unidades"].sum())
print(f"\nSuma de esas 3 filas: {rag}  ·  respuesta real (SQL sobre todas las tiendas): {real}")

,tienda,categoria,mes,unidades,monto
0,Lima,hogar,julio,380,68400
1,Lima,hogar,agosto,265,47700
2,Lima,hogar,septiembre,286,51480
3,Lima,electrónica,julio,362,868800


Filas que trae el RAG de texto (top-3):
  · En septiembre, la tienda de Arequipa vendió 219 unidades de electrónica por 525600 soles.
  · En agosto, la tienda de Arequipa vendió 41 unidades de electrónica por 98400 soles.
  · En julio, la tienda de Arequipa vendió 368 unidades de electrónica por 883200 soles.

Suma de esas 3 filas: 628  ·  respuesta real (SQL sobre todas las tiendas): 678


El RAG recupera filas «parecidas» y **no puede sumar las que no trajo**. Para datos tabulares la herramienta es SQL.

## 2. Un router entrenado con pocas frases

Cuatro rutas: `sql`, `grafo`, `codigo` (palabra clave) y `texto` (híbrido). Entrenamos TF-IDF de caracteres + regresión logística
con ejemplos escritos a mano y probamos con preguntas nuevas.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# ESQUEMA · ENTRENAMIENTO (28 frases de ejemplo; tuplas de 2 campos) → (pregunta, ruta)
#   ruta es la etiqueta que el router debe aprender: sql · grafo · codigo · texto.
ENTRENAMIENTO = [
    # sql: agregaciones sobre la tabla de ventas
    ("¿Cuántas unidades se vendieron en Lima en julio?", "sql"),
    ("Total de ventas en soles de línea blanca en Arequipa", "sql"),
    ("¿Cuál es el promedio de unidades de hogar por mes en Cusco?", "sql"),
    ("¿Qué tienda vendió más electrónica en agosto?", "sql"),
    ("Suma de monto vendido en septiembre", "sql"),
    ("¿Cuántas unidades de hogar se vendieron en total?", "sql"),
    # grafo: relaciones entre personas, equipos, sistemas y proveedores
    ("¿Quién es el jefe de la persona que lidera el equipo de facturación?", "grafo"),
    ("¿De qué proveedor depende el sistema que mantiene el equipo de Pagos?", "grafo"),
    ("¿A quién reporta quien gestiona al proveedor de transporte?", "grafo"),
    ("¿Quién aprueba los viajes de la gerente a la que reporta Marco Salas?", "grafo"),
    ("¿Qué equipo mantiene el sistema que depende de RiskGuard?", "grafo"),
    ("¿Quién supervisa a la administradora de la tienda de Cusco?", "grafo"),
    # codigo: un identificador exacto
    ("ERR-4410", "codigo"), ("SKU-10234", "codigo"), ("CT-2025-0100", "codigo"),
    ("REC-NIGHT-07", "codigo"), ("RB-118", "codigo"), ("ERR-9001", "codigo"),
    ("FAC-2026-1182", "codigo"), ("SKU-55001", "codigo"),
    # texto: preguntas de políticas en lenguaje natural
    ("¿Puedo trabajar desde casa algunos días?", "texto"),
    ("¿Cuántos días de descanso tengo al año?", "texto"),
    ("¿Cómo pido el reembolso de un viaje?", "texto"),
    ("¿Qué hago si olvidé mi contraseña?", "texto"),
    ("¿Me devuelven el dinero si el producto no me sirve?", "texto"),
    ("¿Hasta cuándo puedo cambiar una compra?", "texto"),
    ("¿Hay presupuesto para capacitarme?", "texto"),
    ("¿Qué pasa si llego tarde al trabajo?", "texto"),
]
PRUEBA = [  # preguntas que el router no vio
    ("¿Cuántas unidades de electrónica se vendieron en Cusco en septiembre?", "sql"),
    ("Monto total vendido por la tienda de Lima en agosto", "sql"),
    ("¿Qué tienda vendió más línea blanca en julio?", "sql"),
    ("¿Quién es la jefa de quien lidera Pagos?", "grafo"),
    ("¿Qué proveedor opera el antifraude del que depende PayHub?", "grafo"),
    ("¿A quién reporta la administradora de la tienda de Arequipa?", "grafo"),
    ("ERR-5201", "codigo"), ("SKU-90171", "codigo"), ("CT-2024-0078", "codigo"),
    ("¿Cuántos días de vacaciones me corresponden?", "texto"),
    ("¿Se puede aceptar un obsequio de un proveedor?", "texto"),
    ("¿Cómo funciona el teletrabajo?", "texto"),
]

# Router = clasificador en dos piezas: (1) TF-IDF cuenta trozos de 2 a 4 letras de la pregunta (analyzer="char_wb") y pesa
# más los poco comunes; (2) regresión logística aprende, con los ejemplos, qué trozos indican cada ruta.
router = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True),
                       LogisticRegression(max_iter=2000, C=10))
router.fit([q for q, _ in ENTRENAMIENTO], [r for _, r in ENTRENAMIENTO])

# Se mide con preguntas que el router NO vio al entrenarse. La tabla cruzada (matriz de confusión) muestra, por ruta real,
# a qué ruta las mandó: lo que queda fuera de la diagonal son errores.
pred = router.predict([q for q, _ in PRUEBA])
real_rutas = [r for _, r in PRUEBA]
exactitud = np.mean(np.array(pred) == np.array(real_rutas))
print(f"Exactitud del router en {len(PRUEBA)} preguntas nuevas: {exactitud:.2f}   (¡ojo! son pocas y se parecen a las de entrenamiento)")
pd.crosstab(pd.Series(real_rutas, name="real"), pd.Series(pred, name="predicha"))

Exactitud del router en 12 preguntas nuevas: 1.00   (¡ojo! son pocas y se parecen a las de entrenamiento)


predicha,codigo,grafo,sql,texto
real,,,,
codigo,3,0,0,0
grafo,0,3,0,0
sql,0,0,3,0
texto,0,0,0,3


## 3. Despacho: cada ruta a su método

- `sql` → un **text-to-SQL de juguete** que solo puede usar columnas y valores de una lista blanca (en producción: el LLM propone el SQL,
  tú lo validas y lo ejecutas con un usuario de **solo lectura**).
- `grafo` → recorrido del notebook 03.
- `codigo` → BM25.
- `texto` → híbrido calibrado del notebook 01 (`α · coseno + (1 − α) · BM25`).

**Qué se implementa aquí:** solo la ruta `sql` (la función `a_sql`) y el clasificador del router. Las rutas `grafo`, `codigo` y `texto` reutilizan lo de los notebooks 03 y 01: se nombran para mostrar el despacho, no se vuelven a ejecutar en este notebook.

In [6]:
def a_sql(pregunta):
    """Traduce una pregunta de ventas a SQL parametrizado usando solo valores conocidos (lista blanca)."""
    p = pregunta.lower()
    # Lista blanca: solo se reconocen valores que ya existen en la tabla. Nada del texto del usuario va dentro del SQL.
    tiendas = [t for t in ["Lima", "Arequipa", "Cusco"] if t.lower() in p]
    cats = [c for c in ["hogar", "electrónica", "línea blanca"] if c in p]
    meses = [m for m in ["julio", "agosto", "septiembre"] if m in p]
    # «soles» o «monto» → suma dinero; en otro caso, unidades
    medida = "monto" if any(x in p for x in ["soles", "monto", "ventas en soles"]) else "unidades"
    filtros, params = [], []
    # Cada filtro usa marcadores «?» y los valores viajan aparte (params): SQL parametrizado, que evita la inyección de SQL.
    for col, vals in [("tienda", tiendas), ("categoria", cats), ("mes", meses)]:
        if vals:
            filtros.append(f"{col} IN ({','.join('?' * len(vals))})")
            params += vals
    donde = f" WHERE {' AND '.join(filtros)}" if filtros else ""
    # «¿qué tienda vendió más…?» → agrupa por tienda y se queda con la mayor; si no, un solo total (suma o promedio)
    if "qué tienda" in p or "que tienda" in p:
        return f"SELECT tienda, SUM({medida}) AS total FROM ventas{donde} GROUP BY tienda ORDER BY total DESC LIMIT 1", params
    agg = "AVG" if "promedio" in p else "SUM"
    return f"SELECT {agg}({medida}) FROM ventas{donde}", params


# Base de datos en memoria: la tabla de ventas se carga ahí y las consultas se ejecutan de verdad.
con = sqlite3.connect(":memory:")
ventas.to_sql("ventas", con, index=False)

CASOS_SQL = [  # (pregunta, valor esperado calculado de forma independiente con pandas)
    ("¿Cuántas unidades de electrónica se vendieron en Cusco en septiembre?",
     int(ventas.query("tienda=='Cusco' and categoria=='electrónica' and mes=='septiembre'")["unidades"].sum())),
    ("Monto total vendido por la tienda de Lima en agosto en soles",
     int(ventas.query("tienda=='Lima' and mes=='agosto'")["monto"].sum())),
    ("¿Cuántas unidades de hogar se vendieron en total?", int(ventas.query("categoria=='hogar'")["unidades"].sum())),
    ("¿Qué tienda vendió más línea blanca en julio?",
     ventas.query("categoria=='línea blanca' and mes=='julio'").groupby("tienda")["unidades"].sum().idxmax()),
]
ok = 0
for q, esperado in CASOS_SQL:
    sql, params = a_sql(q)                              # 1) traducir la pregunta a SQL + parámetros
    fila = con.execute(sql, params).fetchone()          # 2) ejecutarlo en la base; 3) comparar con el valor de pandas
    obtenido = fila[0] if len(fila) == 1 else fila[0]
    ok += obtenido == esperado
    print(f"{q}\n   SQL: {sql}  {params}\n   → {obtenido}   (esperado {esperado})  {'✔' if obtenido == esperado else '✘'}")
print(f"\n{ok}/{len(CASOS_SQL)} correctas")

¿Cuántas unidades de electrónica se vendieron en Cusco en septiembre?
   SQL: SELECT SUM(unidades) FROM ventas WHERE tienda IN (?) AND categoria IN (?) AND mes IN (?)  ['Cusco', 'electrónica', 'septiembre']
   → 140   (esperado 140)  ✔
Monto total vendido por la tienda de Lima en agosto en soles
   SQL: SELECT SUM(monto) FROM ventas WHERE tienda IN (?) AND mes IN (?)  ['Lima', 'agosto']
   → 872800   (esperado 872800)  ✔
¿Cuántas unidades de hogar se vendieron en total?
   SQL: SELECT SUM(unidades) FROM ventas WHERE categoria IN (?)  ['hogar']
   → 2198   (esperado 2198)  ✔
¿Qué tienda vendió más línea blanca en julio?
   SQL: SELECT tienda, SUM(unidades) AS total FROM ventas WHERE categoria IN (?) AND mes IN (?) GROUP BY tienda ORDER BY total DESC LIMIT 1  ['línea blanca', 'julio']
   → Lima   (esperado Lima)  ✔

4/4 correctas


In [7]:
assert exactitud >= 0.75, "el router debería acertar al menos 3 de cada 4 preguntas nuevas"
assert ok == len(CASOS_SQL), "el SQL de juguete debería coincidir con pandas en los casos de prueba"
assert rag != real, "el RAG de texto no debería poder reproducir la suma real"
print("Router, SQL y contraste con el RAG de texto verificados")

Router, SQL y contraste con el RAG de texto verificados


## Qué te llevas

- **Los números no se buscan, se calculan.** Si tu pregunta es un conteo, una suma o un máximo, usa SQL; el RAG de texto alucina un resultado plausible.
- **El router es parte del sistema**: se entrena, se mide y se vigila como cualquier otro componente. Con pocas rutas, un clasificador chico basta.
  El 100 % de este notebook sale de 12 preguntas parecidas a las de entrenamiento: es una demostración, no una garantía. En producción mídelo con preguntas reales y registra las que cayeron en la ruta equivocada.
- **Seguridad del SQL generado:** lista blanca de tablas/columnas, parámetros (nunca concatenar texto del usuario), usuario de solo lectura, límite de filas.
- Esta pieza completa el mapa: *texto → híbrido · códigos → palabra clave · relaciones → grafo · números → SQL · documentos largos → árbol*.

➡️ Último: `05_hibrido_a_produccion.ipynb` — lo que separa un demo de un despliegue.

---
*Webinar «RAG para Soluciones Productivas» · Manuel Argüelles · datos 100 % sintéticos de «Andina Retail»: las cifras ilustran un criterio de decisión, no miden tu sistema. Mide siempre con tus propios datos.*